In [2]:
# NB_02_FHIR_Silver - Final Production Version
# ============================================================
# Pure PySpark SCD Type 2 — no SQL subqueries
# Compatible with ALL Microsoft Fabric Delta versions
# ============================================================

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY")
print("✅ Datetime rebase config applied (LEGACY mode)\n")

from pyspark.sql.functions import (
    col, to_timestamp, when, lit, current_timestamp,
    md5, concat_ws, coalesce, row_number, desc
)
from pyspark.sql.types import BooleanType, TimestampType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime


# ─────────────────────────────────────────────
# UTILITY: Safe column existence check
# ─────────────────────────────────────────────
def has_column(df, col_name: str) -> bool:
    return col_name in [field.name for field in df.schema.fields]


# ─────────────────────────────────────────────
# UTILITY: Row hash for change detection
# ─────────────────────────────────────────────
def add_row_hash(df, columns: list):
    existing = [c for c in columns if has_column(df, c)]
    return df.withColumn(
        "row_hash",
        md5(concat_ws(
            "||",
            *[coalesce(col(c).cast("string"), lit("")) for c in existing]
        ))
    )


# ─────────────────────────────────────────────
# UTILITY: Bootstrap — first run only
# ─────────────────────────────────────────────
def bootstrap_silver_table(silver_df, silver_table: str, hash_columns: list):
    """
    Creates silver table with row_hash + SCD columns.
    Called ONLY when table does not exist.
    Never drops existing data.
    """
    now_ts = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S")

    init_df = add_row_hash(silver_df, hash_columns) \
        .withColumn("valid_from",            lit(now_ts).cast(TimestampType())) \
        .withColumn("valid_to",              lit(None).cast(TimestampType())) \
        .withColumn("is_current",            lit(True).cast(BooleanType())) \
        .withColumn("silver_load_timestamp", current_timestamp())

    init_df.write.mode("overwrite").format("delta").saveAsTable(silver_table)
    count = spark.table(silver_table).count()
    print(f"   🆕 Bootstrap: '{silver_table}' created — {count:,} rows ✅")


# ─────────────────────────────────────────────
# CORE: SCD Type 2 MERGE
# Pure PySpark — no SQL subqueries
# Works on ALL Fabric Delta versions
# ─────────────────────────────────────────────
def scd2_merge(silver_table: str, incoming_df, hash_columns: list):
    """
    True SCD Type 2:
      Changed records → expire old + insert new
      New records     → insert
      Unchanged       → do nothing
    """
    now_ts = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S")

    # Prepare incoming with hash + SCD columns
    incoming_hashed = add_row_hash(incoming_df, hash_columns) \
        .withColumn("valid_from",            lit(now_ts).cast(TimestampType())) \
        .withColumn("valid_to",              lit(None).cast(TimestampType())) \
        .withColumn("is_current",            lit(True).cast(BooleanType())) \
        .withColumn("silver_load_timestamp", current_timestamp())

    # ── Step 1: Find changed IDs ──────────────────────────────────
    current_silver = spark.table(silver_table) \
        .filter(col("is_current") == True) \
        .select("id", "row_hash") \
        .withColumnRenamed("row_hash", "existing_hash")

    changed_ids_df = incoming_hashed \
        .select("id", "row_hash") \
        .join(current_silver, on="id", how="inner") \
        .filter(col("row_hash") != col("existing_hash")) \
        .select("id")

    changed_ids = [row["id"] for row in changed_ids_df.collect()]
    print(f"   🔍 Found {len(changed_ids)} changed record(s)")

    # ── Step 2: Expire changed rows ───────────────────────────────
    if changed_ids:
        delta_table = DeltaTable.forName(spark, silver_table)
        delta_table.update(
            condition=(col("is_current") == True) & (col("id").isin(changed_ids)),
            set={
                "valid_to":   lit(now_ts).cast(TimestampType()),
                "is_current": lit(False).cast(BooleanType())
            }
        )
        print(f"   ✅ Step 1 — expired {len(changed_ids)} changed row(s)")
    else:
        print(f"   ✅ Step 1 — no rows to expire")

    # ── Step 3: Insert new + changed rows ────────────────────────
    # New records — id never seen in silver before
    all_silver_ids = spark.table(silver_table).select("id").distinct()
    new_rows       = incoming_hashed.join(all_silver_ids, on="id", how="left_anti")

    # Changed records — just expired above
    if changed_ids:
        changed_rows = incoming_hashed.filter(col("id").isin(changed_ids))
    else:
        changed_rows = spark.createDataFrame([], incoming_hashed.schema)

    rows_to_insert = new_rows.union(changed_rows)
    insert_count   = rows_to_insert.count()

    if insert_count > 0:
        rows_to_insert.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(silver_table)
        print(f"   ✅ Step 2 — inserted {insert_count:,} new/changed row(s)")
    else:
        print(f"   ✔️  No changes — silver is already up to date")


# ─────────────────────────────────────────────
# MAIN: Transform Bronze → Silver
# ─────────────────────────────────────────────
def transform_to_silver(resource_type: str):
    bronze_table = f"bronze_{resource_type.lower()}"
    silver_table = f"silver_{resource_type.lower()}"

    print(f"\n{'─'*70}")
    print(f"🔄  {bronze_table}  →  {silver_table}")
    print(f"{'─'*70}")

    raw_df = spark.table(bronze_table)

    window  = Window.partitionBy("id").orderBy(desc("extraction_timestamp"))
    deduped = raw_df \
        .withColumn("_rn", row_number().over(window)) \
        .filter(col("_rn") == 1) \
        .drop("_rn") \
        .withColumn("resource_type", lit(resource_type))

    if resource_type == "Patient":
        if has_column(deduped, "birthDate"):
            deduped = deduped.withColumn(
                "birthDate",
                when(col("birthDate").isNotNull(),
                     to_timestamp(col("birthDate"), "yyyy-MM-dd"))
            )
        silver_df = deduped.select(
            "id", "resource_type", "gender", "birthDate",
            col("name").alias("patient_name"), "address",
            "extraction_timestamp", "load_date",
            "api_base_url", "api_params"
        )
        hash_cols = ["id", "gender", "birthDate"]

    elif resource_type == "Encounter":
        if has_column(deduped, "period"):
            deduped = deduped \
                .withColumn("period_start",
                    when(col("period").isNotNull(),
                         to_timestamp(col("period").getField("start")))) \
                .withColumn("period_end",
                    when(col("period").isNotNull(),
                         to_timestamp(col("period").getField("end"))))
        silver_df = deduped.select(
            "id", "resource_type", "status", "class",
            "period_start", "period_end", "subject", "type",
            "extraction_timestamp", "load_date",
            "api_base_url", "api_params"
        )
        hash_cols = ["id", "status", "period_start", "period_end"]

    elif resource_type == "Observation":
        if has_column(deduped, "effectiveDateTime"):
            deduped = deduped.withColumn(
                "effectiveDateTime",
                when(col("effectiveDateTime").isNotNull(),
                     to_timestamp(col("effectiveDateTime")))
            )
        silver_df = deduped.select(
            "id", "resource_type", "status", "code",
            "valueQuantity", "effectiveDateTime", "subject", "encounter",
            "extraction_timestamp", "load_date",
            "api_base_url", "api_params"
        )
        hash_cols = ["id", "status", "effectiveDateTime", "valueQuantity"]

    elif resource_type == "Condition":
        for ts_col in ["onsetDateTime", "recordedDate"]:
            if has_column(deduped, ts_col):
                deduped = deduped.withColumn(
                    ts_col,
                    when(col(ts_col).isNotNull(), to_timestamp(col(ts_col)))
                )
        silver_df = deduped.select(
            "id", "resource_type", "clinicalStatus", "code",
            "subject", "encounter", "onsetDateTime", "recordedDate",
            "extraction_timestamp", "load_date",
            "api_base_url", "api_params"
        )
        hash_cols = ["id", "clinicalStatus", "onsetDateTime", "recordedDate"]

    else:
        raise ValueError(f"Unknown resource_type: {resource_type}")

    # Bootstrap first time / Merge every time after
    table_exists = spark.catalog.tableExists(silver_table)

    if not table_exists:
        print(f"   ℹ️  First run — bootstrapping...")
        bootstrap_silver_table(silver_df, silver_table, hash_cols)
    else:
        print(f"   ℹ️  Table exists — running SCD Type 2 merge...")
        scd2_merge(silver_table, silver_df, hash_cols)

    # Summary
    final           = spark.table(silver_table)
    total_rows      = final.count()
    current_rows    = final.filter(col("is_current") == True).count()
    historical_rows = total_rows - current_rows

    print(f"\n   📊 '{silver_table}' summary:")
    print(f"      Total rows      : {total_rows:,}")
    print(f"      Current rows    : {current_rows:,}   ← is_current = True")
    print(f"      Historical rows : {historical_rows:,}   ← is_current = False")

    display(final.filter(col("is_current") == True).limit(5))
    return silver_table


# ═══════════════════════════════════════════════════════════════════
# ENTRY POINT
# ═══════════════════════════════════════════════════════════════════
print("=" * 70)
print("🏗️   SILVER LAYER — SCD TYPE 2 TRANSFORMATION")
print("=" * 70)

results = {}
for resource in ["Patient", "Encounter", "Observation", "Condition"]:
    try:
        transform_to_silver(resource)
        results[resource] = "✅ SUCCESS"
    except Exception as e:
        results[resource] = f"❌ FAILED: {e}"
        print(f"\n❌ Error on {resource}: {e}")

print("\n" + "=" * 70)
print("📋  SILVER LAYER RUN SUMMARY")
print("=" * 70)
for resource, status in results.items():
    print(f"   {resource:<15} → {status}")


# ═══════════════════════════════════════════════════════════════════
# VERIFICATION
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("🔍  SCD TYPE 2 VERIFICATION")
print("=" * 70)

for resource in ["Patient", "Encounter", "Observation", "Condition"]:
    silver_table = f"silver_{resource.lower()}"
    try:
        df         = spark.table(silver_table)
        total      = df.count()
        current    = df.filter(col("is_current") == True).count()
        historical = total - current
        has_hash   = "row_hash" in [f.name for f in df.schema.fields]

        print(f"\n   {silver_table}")
        print(f"      Total      : {total:,}")
        print(f"      Current    : {current:,}")
        print(f"      Historical : {historical:,}  {'🔄 SCD Type 2 working!' if historical > 0 else '✔️  No changes yet'}")
        print(f"      row_hash   : {'✅ EXISTS' if has_hash else '❌ MISSING'}")

    except Exception as e:
        print(f"   ❌ {silver_table}: {e}")

StatementMeta(, a32c999b-6951-43bc-bb26-eb396de9edd6, 4, Finished, Available, Finished, False)

✅ Datetime rebase config applied (LEGACY mode)

🏗️   SILVER LAYER — SCD TYPE 2 TRANSFORMATION

──────────────────────────────────────────────────────────────────────
🔄  bronze_patient  →  silver_patient
──────────────────────────────────────────────────────────────────────
   ℹ️  Table exists — running SCD Type 2 merge...
   🔍 Found 1 changed record(s)
   ✅ Step 1 — expired 1 changed row(s)
   ✅ Step 2 — inserted 1 new/changed row(s)

   📊 'silver_patient' summary:
      Total rows      : 1,331
      Current rows    : 1,330   ← is_current = True
      Historical rows : 1   ← is_current = False


SynapseWidget(Synapse.DataFrame, ded52cf1-e14e-4f48-94e3-921fbe2ad6c4)


──────────────────────────────────────────────────────────────────────
🔄  bronze_encounter  →  silver_encounter
──────────────────────────────────────────────────────────────────────
   ℹ️  Table exists — running SCD Type 2 merge...
   🔍 Found 0 changed record(s)
   ✅ Step 1 — no rows to expire
   ✔️  No changes — silver is already up to date

   📊 'silver_encounter' summary:
      Total rows      : 1,502
      Current rows    : 1,502   ← is_current = True
      Historical rows : 0   ← is_current = False


SynapseWidget(Synapse.DataFrame, c4582a69-1f69-4609-91cb-08bbe21aed7a)


──────────────────────────────────────────────────────────────────────
🔄  bronze_observation  →  silver_observation
──────────────────────────────────────────────────────────────────────
   ℹ️  Table exists — running SCD Type 2 merge...
   🔍 Found 0 changed record(s)
   ✅ Step 1 — no rows to expire
   ✔️  No changes — silver is already up to date

   📊 'silver_observation' summary:
      Total rows      : 1,500
      Current rows    : 1,500   ← is_current = True
      Historical rows : 0   ← is_current = False


SynapseWidget(Synapse.DataFrame, f609e390-a672-4965-96d0-2e72e5bf114b)


──────────────────────────────────────────────────────────────────────
🔄  bronze_condition  →  silver_condition
──────────────────────────────────────────────────────────────────────
   ℹ️  Table exists — running SCD Type 2 merge...
   🔍 Found 0 changed record(s)
   ✅ Step 1 — no rows to expire
   ✔️  No changes — silver is already up to date

   📊 'silver_condition' summary:
      Total rows      : 1,500
      Current rows    : 1,500   ← is_current = True
      Historical rows : 0   ← is_current = False


SynapseWidget(Synapse.DataFrame, cdd65f79-3fad-4fc5-8452-987f2ac24662)


📋  SILVER LAYER RUN SUMMARY
   Patient         → ✅ SUCCESS
   Encounter       → ✅ SUCCESS
   Observation     → ✅ SUCCESS
   Condition       → ✅ SUCCESS

🔍  SCD TYPE 2 VERIFICATION

   silver_patient
      Total      : 1,331
      Current    : 1,330
      Historical : 1  🔄 SCD Type 2 working!
      row_hash   : ✅ EXISTS

   silver_encounter
      Total      : 1,502
      Current    : 1,502
      Historical : 0  ✔️  No changes yet
      row_hash   : ✅ EXISTS

   silver_observation
      Total      : 1,500
      Current    : 1,500
      Historical : 0  ✔️  No changes yet
      row_hash   : ✅ EXISTS

   silver_condition
      Total      : 1,500
      Current    : 1,500
      Historical : 0  ✔️  No changes yet
      row_hash   : ✅ EXISTS
